# Pixal3D CUDA · T4 operator profiling
This notebook measures **synthetic warm operators**, not pretrained inference, GLB quality or end-to-end speedup. It downloads Python packages/source but **no pretrained model weights** and needs **no Hugging Face token or user photos**.
Select a fresh **T4 GPU** runtime. Do not use a CPU/L4 result as a T4 measurement. Installation/download time is outside the operator timers. Profiling adds overhead and traces capture only one representative call per case.
See [scope, correctness checks and acceptance requirements](https://github.com/syi0808/pixal3d-cuda/blob/main/docs/performance.md). Runtime precision, 512 geometry and 12-step defaults have not been lowered. No Modal resource is created.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

def run_command(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

if "torch" in sys.modules:
    raise RuntimeError("Restart the runtime before installing PyTorch.")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
ROOT = Path("/content/pixal3d-cuda-perf")
if not ROOT.exists():
    run_command(["git", "clone", "--depth", "1", "--branch", "main",
                 "https://github.com/syi0808/pixal3d-cuda.git", str(ROOT)])
os.chdir(ROOT)
print("Source:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
run_command([sys.executable, "-m", "pip", "install", "torch==2.6.0", "torchvision==0.21.0",
             "--index-url", "https://download.pytorch.org/whl/cu124"])
run_command([sys.executable, "-m", "pip", "install", "-e", f"{ROOT}[dev]"])

In [ ]:
import torch

assert torch.cuda.is_available(), "Select a real T4 GPU runtime."
assert "T4" in torch.cuda.get_device_name() and torch.cuda.get_device_capability() == (7, 5), torch.cuda.get_device_name()
print(subprocess.check_output(["nvidia-smi"], text=True))
run_command([sys.executable, "-m", "pixalcuda.cli", "doctor"])
run_command([sys.executable, "-m", "pytest", "-q", "tests/test_performance_paths.py",
             "tests/test_sparse.py", "tests/test_ops.py", "tests/test_weights_runtime.py"])

## Warm measurements and representative traces
The sweeps compare attention query chunks/layouts, sparse convolution layouts, neighbor-map construction, coordinate-only early exit and static projection reuse.
Timing includes each candidate's required layout conversion/precompute, but does not include profiler overhead. It is **not a complete 30-block pretrained flow**. Keep all arguments and seeds identical when comparing runs. OOM or numerical mismatch fails rather than reducing the request.
The default 4096 tokens are a starting point; test larger cases only after inspecting memory. These measurements consume Colab GPU time.

In [ ]:
OUTPUT = ROOT / "outputs" / "t4-operators"
OUTPUT.mkdir(parents=True, exist_ok=True)
run_command([sys.executable, "-u", "scripts/benchmark_cuda_ops.py",
             "--device", "cuda", "--tokens", "4096", "--channels", "1536", "--heads", "12",
             "--query-chunks", "256", "512", "1024", "2048",
             "--warmup", "2", "--repeats", "5",
             "--output", str(OUTPUT / "operators.json"), "--trace-dir", str(OUTPUT / "traces")])

## Optional: explicitly test SDPA backends
Set `RUN_BACKEND_SWEEP=True` to compare forced backends. Unsupported FP32/T4 kernels are recorded as **unsupported**, not replaced with FP16 or another backend. Automatic dispatch is preserved in the normal runtime.
Check observed `sdpa_operators` and `cuda_activity_captured`; CUDA-event timings alone do not prove CUPTI kernel traces were captured.

In [ ]:
RUN_BACKEND_SWEEP = False
if RUN_BACKEND_SWEEP:
    run_command([sys.executable, "-u", "scripts/benchmark_cuda_ops.py",
                 "--device", "cuda", "--cases", "attention", "--tokens", "4096",
                 "--sdpa-backends", "auto", "math", "efficient", "flash",
                 "--query-chunks", "512", "--warmup", "2", "--repeats", "3",
                 "--output", str(OUTPUT / "backends.json"), "--trace-dir", str(OUTPUT / "backend-traces")])

In [ ]:
import json

report = json.loads((OUTPUT / "operators.json").read_text())
assert report["status"] == "complete", report.get("error")
print("Actual GPU:", report["gpu"], report["capability"], "torch:", report["torch"])
for case in report["cases"]:
    print(case["name"], case["status"],
          "wall ms:", case.get("median_wall_ms"),
          "CUDA interval ms:", case.get("median_cuda_interval_ms"),
          "incremental peak MiB:", case.get("incremental_peak_allocated_bytes", 0) / 1024**2,
          "max absolute error:", case.get("max_abs_error"))
    if "trace" in case:
        print("Observed SDPA:", case["trace"]["sdpa_operators"],
              "CUDA trace captured:", case["trace"]["cuda_activity_captured"])
print("Synthetic results are not a pretrained T4 GLB acceptance or quality-parity result.")

## Download evidence and stop the runtime
Download the timing JSON and trace archive. Open traces in a trace viewer; L1/L2/DRAM/occupancy counters require separate Nsight Compute profiling. When finished, disconnect/delete the Colab runtime to stop GPU usage.

In [ ]:
import shutil
from google.colab import files

files.download(str(OUTPUT / "operators.json"))
archive = shutil.make_archive("/content/pixal3d-t4-operators", "zip", root_dir=OUTPUT)
files.download(archive)